# Nirvana - Disaster Relief Predictive Modeling

This notebook demonstrates the training and evaluation of machine learning models to forecast disaster relief demand in coastal Odisha.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import LinearRegression

# Load data
df = pd.read_csv('../data/raw/historical_disaster_data.csv')
df['date'] = pd.to_datetime(df['date'])
df.head()

,date,district,current_rainfall,rainfall_3day,rainfall_7day,rainfall_anomaly,cyclone_intensity,wind_speed,distance_from_coastline,elevation,...,population_density,vulnerable_population,rural_population_pct,hospitals,shelters,affected_population,food_demand,water_demand,medical_demand,shelter_demand
0,2015-07-11,Kendrapara,292.942847,729.518488,1418.272832,2.929428,2,101.147697,48,12,...,544.629349,1353600,0.94,45,120,207119,395862,846868,21620,39355
1,2015-07-11,Jagatsinghpur,289.835013,711.753744,1422.584192,2.898350,2,108.825245,45,15,...,681.055156,1011040,0.89,38,110,157665,306903,604841,15953,31162
2,2015-07-11,Puri,315.888267,802.231002,1621.064430,3.158883,2,107.066009,50,10,...,488.071285,1426320,0.84,55,150,260772,524844,1031135,24814,50750
3,2015-07-11,Ganjam,324.091924,777.196878,1676.971031,3.240919,2,113.731189,35,25,...,430.051182,2752620,0.78,120,250,403731,798665,1563607,41404,80141
4,2015-07-11,Balasore,298.281651,680.354100,1439.213435,2.982817,2,111.726937,44,16,...,609.563847,2064800,0.89,75,180,306892,618107,1230125,32104,63493


### Feature Engineering

In [2]:
df_processed = pd.get_dummies(df, columns=['district'], drop_first=False)
df_processed['month'] = df_processed['date'].dt.month
df_processed = df_processed.drop(columns=['date'])
df_processed.head()

,current_rainfall,rainfall_3day,rainfall_7day,rainfall_anomaly,cyclone_intensity,wind_speed,distance_from_coastline,elevation,flood_prone_indicator,population_density,...,district_Balasore,district_Bhadrak,district_Cuttack,district_Ganjam,district_Jagatsinghpur,district_Jajpur,district_Kendrapara,district_Khordha,district_Puri,month
0,292.942847,729.518488,1418.272832,2.929428,2,101.147697,48,12,1,544.629349,...,False,False,False,False,False,False,True,False,False,7
1,289.835013,711.753744,1422.584192,2.898350,2,108.825245,45,15,1,681.055156,...,False,False,False,False,True,False,False,False,False,7
2,315.888267,802.231002,1621.064430,3.158883,2,107.066009,50,10,1,488.071285,...,False,False,False,False,False,False,False,False,True,7
3,324.091924,777.196878,1676.971031,3.240919,2,113.731189,35,25,0,430.051182,...,False,False,False,True,False,False,False,False,False,7
4,298.281651,680.354100,1439.213435,2.982817,2,111.726937,44,16,1,609.563847,...,True,False,False,False,False,False,False,False,False,7


### Model Training and Evaluation

In [3]:
targets = ['affected_population', 'food_demand', 'water_demand', 'medical_demand', 'shelter_demand']

for target_name in targets:
    print(f"\n--- Evaluating {target_name} ---")
    X = df_processed.drop(columns=targets)
    y = df_processed[target_name]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    models = {
        'Linear Regression': LinearRegression(),
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
        'XGBoost': XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
        'LightGBM': LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbose=-1)
    }
    
    best_model = None
    best_mape = float('inf')
    
    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = np.maximum(0, model.predict(X_test))
        
        mape = mean_absolute_percentage_error(y_test + 1e-5, preds)
        print(f"{name} MAPE: {mape*100:.2f}%")
        
        if mape < best_mape and name != 'Linear Regression':
            best_mape = mape
            best_model = name
            
    print(f"Best model: {best_model} with {best_mape*100:.2f}% MAPE. Passes Success Metric (<20%).")


--- Evaluating affected_population ---
Linear Regression MAPE: 20.61%
Random Forest MAPE: 16.97%


XGBoost MAPE: 14.60%
LightGBM MAPE: 31.33%
Best model: XGBoost with 14.60% MAPE. Passes Success Metric (<20%).

--- Evaluating food_demand ---
Linear Regression MAPE: 20.41%
Random Forest MAPE: 17.52%
XGBoost MAPE: 14.69%
LightGBM MAPE: 30.75%
Best model: XGBoost with 14.69% MAPE. Passes Success Metric (<20%).

--- Evaluating water_demand ---
Linear Regression MAPE: 20.93%


Random Forest MAPE: 17.66%
XGBoost MAPE: 15.74%
LightGBM MAPE: 30.02%
Best model: XGBoost with 15.74% MAPE. Passes Success Metric (<20%).

--- Evaluating medical_demand ---
Linear Regression MAPE: 21.05%
Random Forest MAPE: 19.17%


XGBoost MAPE: 15.20%
LightGBM MAPE: 29.83%
Best model: XGBoost with 15.20% MAPE. Passes Success Metric (<20%).

--- Evaluating shelter_demand ---
Linear Regression MAPE: 22.58%
Random Forest MAPE: 18.07%
XGBoost MAPE: 15.20%
LightGBM MAPE: 30.21%
Best model: XGBoost with 15.20% MAPE. Passes Success Metric (<20%).
